## Importar o CSV

In [2]:
import pandas as pd

cols = ['NU_INSCRICAO', 'IN_TREINEIRO', 'TP_SEXO', 'TP_FAIXA_ETARIA',
        'TP_COR_RACA', 'TP_ESTADO_CIVIL', 'TP_ST_CONCLUSAO', 'TP_ANO_CONCLUIU',
        'TP_ENSINO', 'CO_MUNICIPIO_PROVA', 'NO_MUNICIPIO_PROVA',
        'CO_UF_PROVA', 'SG_UF_PROVA', 'Q001', 'Q002', 'Q003', 'Q004',
        'Q005', 'Q006', 'Q020', 'Q023']

df = pd.read_csv(
    '/home/gabriel/Documentos/AnaliseDadosBigData/microdados_enem_2025/DADOS/PARTICIPANTES_2025.csv',
    sep=';',
    encoding='ISO-8859-1',
    usecols=cols
)


In [3]:
df

,NU_INSCRICAO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ENSINO,IN_TREINEIRO,CO_MUNICIPIO_PROVA,...,CO_UF_PROVA,SG_UF_PROVA,Q001,Q002,Q003,Q004,Q005,Q006,Q020,Q023
0,210066506229,6,F,1,2,1,4,NaN,0,2932200,...,29,BA,C,F,A,A,1,B,A,A
1,210066506230,3,M,1,1,2,0,1.0,0,2910800,...,29,BA,B,E,C,B,3,A,B,A
2,210066506231,2,F,1,1,3,0,NaN,1,3549805,...,35,SP,E,E,D,D,6,A,B,C
3,210066506232,2,M,1,3,2,0,1.0,0,2203909,...,22,PI,D,F,D,D,3,A,B,D
4,210066506233,1,F,1,1,3,0,NaN,1,5101803,...,51,MT,G,G,E,E,4,B,B,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4810767,210071444509,12,F,1,1,1,14,NaN,0,2611606,...,26,PE,G,G,E,E,7,B,B,D
4810768,210071444513,4,M,1,1,1,1,NaN,0,3522208,...,35,SP,F,E,D,B,4,A,B,E
4810769,210071444514,11,M,1,1,1,10,NaN,0,3505708,...,35,SP,E,E,F,F,4,B,B,A
4810770,210071444515,8,F,1,2,1,5,NaN,0,3509502,...,35,SP,E,E,F,F,1,B,B,A


---

## Filtrar para trocar os Nulos e Treineiros

In [4]:
df['IN_TREINEIRO'].value_counts() #Saber quantos tem
df = df[df['IN_TREINEIRO'] == 0].copy() #Filtra só para não ter treineiros


In [5]:
df.isnull().sum().sort_values(ascending=False) #Ver os nulos por coluna do maior ao menor
#Somento o TP_ENSINO ta(tava) com 2092958 valores nulos

TP_ENSINO             2092958
TP_FAIXA_ETARIA             0
TP_SEXO                     0
TP_ESTADO_CIVIL             0
NU_INSCRICAO                0
TP_COR_RACA                 0
TP_ST_CONCLUSAO             0
TP_ANO_CONCLUIU             0
IN_TREINEIRO                0
CO_MUNICIPIO_PROVA          0
NO_MUNICIPIO_PROVA          0
CO_UF_PROVA                 0
SG_UF_PROVA                 0
Q001                        0
Q002                        0
Q003                        0
Q004                        0
Q005                        0
Q006                        0
Q020                        0
Q023                        0
dtype: int64

In [6]:
#Colunas de identifição e localização
cols_essenciais = ['CO_MUNICIPIO_PROVA', "SG_UF_PROVA", "TP_SEXO"]
df = df.dropna(subset=cols_essenciais)

#Colunas de questionario socioeconomico
cols_questionario = ['Q001', "Q002", "Q003", "Q004", "Q005", "Q006"]
for col in cols_questionario:
    df[col] = df[col].fillna("NÃO INFORMADO")

#Categorias das caracteristicas do candiadato
cols_tp = ['TP_COR_RACA', 'TP_ESTADO_CIVIL', 'TP_ST_CONCLUSAO', 'TP_ANO_CONCLUIU', 'TP_ENSINO']
for col in cols_tp:
    df[col] = df[col].fillna('NÃO INFORMADO')

---

## Participação por município/UF

In [7]:
part_municipio = (
    df.groupby(["NO_MUNICIPIO_PROVA", "SG_UF_PROVA"]).size()
    .reset_index(name='qtd_participantes')
    .sort_values('qtd_participantes' , ascending=False)
)

part_municipio.head(10)


,NO_MUNICIPIO_PROVA,SG_UF_PROVA,qtd_participantes
1610,São Paulo,SP,160867
1383,Rio de Janeiro,RJ,110561
1404,Salvador,BA,72167
260,Brasília,DF,67665
603,Fortaleza,CE,65840
212,Belo Horizonte,MG,59170
949,Manaus,AM,57916
215,Belém,PA,56554
1593,São Luís,MA,45548
1345,Recife,PE,42009


In [8]:
"""
Pensando em inisght pra cá, talvez junte com o Urbano X Rural para enriquecer mais aqui
"""
part_municipio.tail(10)

,NO_MUNICIPIO_PROVA,SG_UF_PROVA,qtd_participantes
1715,Uiramutã,RR,104
392,Carlinda,MT,101
95,Araguanã,TO,101
184,Barra do Turvo,SP,99
1147,Pacaraima,RR,89
264,Brejinho de Nazaré,TO,70
51,Amajari,RR,53
1440,Santa Rosa do Purus,AC,48
588,Fernando de Noronha,PE,44
587,Fernando Falcão,MA,32


In [9]:
part_uf = df["SG_UF_PROVA"].value_counts().reset_index()
part_uf.columns = ["UF", "qtd_participantes"]

part_uf.head(5)

,UF,qtd_participantes
0,SP,603948
1,MG,368958
2,BA,342861
3,RJ,275221
4,PA,228700


In [10]:
part_uf.tail(5)

,UF,qtd_participantes
22,RO,38576
23,TO,27323
24,AP,27263
25,AC,24447
26,RR,11212


In [11]:
#Grafico com matplos

---


## O Recorte Racial

In [35]:
# Ta como numeros os valores, vou trocar para valores que estão relacionados baseado no dicionario (lembrar de fazer isso nas outras que tiver) pra melhorar a leitura
df['TP_COR_RACA'].value_counts()

TP_COR_RACA
3    1731462
1    1443406
2     518900
4      52684
0      44603
5      32067
Name: count, dtype: int64

In [36]:
#Mapear os valores
mapa_cor_raca = {
    0:'Não Declarado',
    1:'Branco',
    2:'Preta',
    3:'Parda',
    4:'Amarela',
    5:'Indigena'
}

#colocar o mapeamento na coluna, atribuimos a outra coluna nova
df['cor_raca'] = df['TP_COR_RACA'].map(mapa_cor_raca)

#Verificando
df['cor_raca'].value_counts()

cor_raca
Parda            1731462
Branco           1443406
Preta             518900
Amarela            52684
Não Declarado      44603
Indigena           32067
Name: count, dtype: int64

In [37]:
# Agora analisar o tipo de escola
df['Q023'].value_counts()

Q023
A    3086940
D     405622
B     136894
E     113909
C      68011
F      11746
Name: count, dtype: int64

In [38]:
# A coluna Q023 (Em que tipo de escola você frequentou ou frequenta o Ensino Médio?) está com mesma formatação da anterior (que eu lembre de limpar todas)
mapa_tipo_escola = {
    'A': 'Só pública',
    'B': 'Pública + privada (sem bolsa)',
    'C': 'Pública + privada (com bolsa)',
    'D': 'Só privada (sem bolsa)',
    'E': 'Só privada (com bolsa)',
    'F': 'Não frequentou Ensino Médio'
}

#mapear ela em nova coluna
df['tipo_escola'] = df["Q023"].map(mapa_tipo_escola)

#Verificando
df['tipo_escola'].value_counts()

tipo_escola
Só pública                       3086940
Só privada (sem bolsa)            405622
Pública + privada (sem bolsa)     136894
Só privada (com bolsa)            113909
Pública + privada (com bolsa)      68011
Não frequentou Ensino Médio        11746
Name: count, dtype: int64

In [39]:
#Agora cruzar as duas colunas para tirar boas conclusões
tabela_raca_escola = pd.crosstab(df['cor_raca'], df['tipo_escola'], normalize='index') * 100
tabela_raca_escola.round(1)

tipo_escola,Não frequentou Ensino Médio,Pública + privada (com bolsa),Pública + privada (sem bolsa),Só privada (com bolsa),Só privada (sem bolsa),Só pública
cor_raca,,,,,,
Amarela,0.3,1.7,3.8,3.1,12.9,78.2
Branco,0.3,2.0,4.4,4.3,18.8,70.2
Indigena,0.3,1.1,2.0,0.9,1.6,94.0
Não Declarado,0.6,1.7,4.2,2.7,10.7,80.1
Parda,0.3,1.6,3.1,2.1,5.8,87.0
Preta,0.3,1.9,2.9,2.2,4.2,88.5


A raça indígena apresenta a maior concentração em escolas exclusivamente públicas (94,0%) e a menor presença em escolas privadas com bolsa integral (0,9%), indicando menor acesso à rede privada em comparação às demais categorias.

Cabe notar que a categoria Indígena representa uma fração pequena da amostra total (n = 32.067), o que deve ser considerado na interpretação da magnitude do resultado

In [40]:
#Fazer grafico com matplot

---


## Candidato Possui internet?
 (Acho que falta algo a mais, pedir ideia pro professor e pro vinicius de coisas que posso botar)

In [ ]:
"""
De acordo com o Dicionario do enem (arquivo esse na pasta /Dicionario/Dicionário_Microdados_Enem_2025.xlsx)
A - Não
B - Sim

Vou mapear para ajustar melhor o resultado

87,62% dos candidatos possuem internet
12,37% não possuem
"""

mapa_internet = {
    'A':'Não',
    'B':'Sim'
}

df['tem_internet'] = df['Q020'].map(mapa_internet)

df['tem_internet'].value_counts(normalize=True) * 100


tem_internet
Sim    87.626709
Não    12.373291
Name: proportion, dtype: float64

In [48]:
#Vamos separar por estado

"""
É notado uma grande deficiencia com internet na nossa nação, observando na tabela onde o estado
do Amazonas , os candidatos que possuem não internet ocupam 31.22% do total de inscritos, isso sem
incluir treineiros.
"""

tabela_internet_uf = pd.crosstab(df['SG_UF_PROVA'], df['tem_internet'], normalize='index') * 100
tabela_internet_uf.round(2).sort_values('Não', ascending=False)



tem_internet,Não,Sim
SG_UF_PROVA,,
AM,31.22,68.78
PA,27.04,72.96
AC,27.03,72.97
MA,23.11,76.89
AP,21.90,78.10
PI,18.30,81.70
SE,17.75,82.25
CE,16.55,83.45
TO,16.03,83.97


---

## **Escolaridade dos pais x tipo de escola do filho**



In [61]:
#separar o mapa aqui pra deixar logo pro Q001 e Q002 por serem mesmas categorias
mapa_responsaveis = {
    'A': 'Nunca estudou',
    'B': 'Fund. incompleto (até 4ª/5º)',
    'C': 'Fund. incompleto (até 8ª/9º)',
    'D': 'Fund. completo',
    'E': 'Médio completo',
    'F': 'Superior completo',
    'G': 'Pós-graduação',
    'H': 'Não sei'
}

In [62]:
#Ate que serie seu pai/homem responsável estudou

df['escolaridade_pai'] = df['Q001'].map(mapa_responsaveis)

df['escolaridade_pai'].value_counts()


escolaridade_pai
Médio completo                  1055981
Fund. incompleto (até 4ª/5º)     651283
Fund. completo                   510593
Não sei                          489499
Fund. incompleto (até 8ª/9º)     450096
Superior completo                277304
Pós-graduação                    202126
Nunca estudou                    186240
Name: count, dtype: int64

In [63]:
#Ate que serie sua mãe/mulher responsável estudou

df['escolaridade_mae'] = df['Q002'].map(mapa_responsaveis)

df['escolaridade_mae'].value_counts()

escolaridade_mae
Médio completo                  1344927
Fund. completo                   561125
Fund. incompleto (até 4ª/5º)     469676
Superior completo                387656
Fund. incompleto (até 8ª/9º)     383991
Pós-graduação                    367053
Não sei                          187107
Nunca estudou                    121587
Name: count, dtype: int64

In [64]:
df['tipo_escola'].value_counts()

tipo_escola
Só pública                       3086940
Só privada (sem bolsa)            405622
Pública + privada (sem bolsa)     136894
Só privada (com bolsa)            113909
Pública + privada (com bolsa)      68011
Não frequentou Ensino Médio        11746
Name: count, dtype: int64

In [84]:
#escolaridade do pai x tipo de escola
pai_tipo = (pd.crosstab(df['escolaridade_pai'], df['tipo_escola'], normalize='index') * 100).round(1)
pai_tipo.sort_values(['Só privada (sem bolsa)', 'Só pública'], ascending=[False,True] )

tipo_escola,Não frequentou Ensino Médio,Pública + privada (com bolsa),Pública + privada (sem bolsa),Só privada (com bolsa),Só privada (sem bolsa),Só pública
escolaridade_pai,,,,,,
Pós-graduação,0.2,1.9,5.9,7.1,49.9,35.1
Superior completo,0.2,2.4,6.2,7.3,36.8,47.1
Médio completo,0.2,2.3,4.7,4.2,12.0,76.7
Fund. completo,0.2,1.9,3.2,2.4,5.5,86.9
Não sei,0.5,1.4,2.9,1.6,4.0,89.6
Fund. incompleto (até 8ª/9º),0.3,1.5,2.8,1.7,3.6,90.1
Fund. incompleto (até 4ª/5º),0.4,1.2,2.0,1.0,1.7,93.8
Nunca estudou,0.8,1.0,1.5,0.7,0.9,95.2


In [ ]:
#escolaridade da mãe x tipo de escola
mae_tipo = (pd.crosstab(df['escolaridade_mae'], df['tipo_escola'], normalize='index') * 100).round(1)
mae_tipo.sort_values(['Só privada (sem bolsa)', 'Só pública'], ascending=[False,True] )

tipo_escola,Não frequentou Ensino Médio,Pública + privada (com bolsa),Pública + privada (sem bolsa),Só privada (com bolsa),Só privada (sem bolsa),Só pública
escolaridade_mae,,,,,,
Pós-graduação,0.1,2.3,5.8,7.0,36.8,48.0
Superior completo,0.2,2.7,5.7,6.7,29.2,55.5
Médio completo,0.2,2.1,4.1,3.3,8.8,81.6
Não sei,0.7,1.1,2.8,1.1,3.7,90.7
Fund. completo,0.3,1.4,2.6,1.5,3.3,90.9
Fund. incompleto (até 8ª/9º),0.4,1.2,2.2,1.1,2.2,93.0
Fund. incompleto (até 4ª/5º),0.5,1.1,1.8,0.7,1.1,94.9
Nunca estudou,1.2,1.0,1.6,0.6,0.8,94.9


na geração dos pais desses candidatos, homem com curso superior tende a ganhar mais que mulher com o mesmo curso (desigualdade salarial de gênero no Brasil), então escolaridade do pai pode ser mais forte pra renda familiar alta do que escolaridade da mãe.

pode ter mais registro de "mãe responsável" em famílias monoparentais (mãe solo), que tendem a ter renda per capita menor mesmo com escolaridade alta, puxando a média de "mãe com pós" pra baixo.

In [ ]:
#Gráfico com matplot

---

## **Situação de conclusão do ensino médio**

In [90]:
mapa_conclusao = {
    1: 'Já concluí o Ensino Médio',
    2: 'Estou cursando e concluirei o Ensino Médio em 2025',
    3: 'Estou cursando e concluirei o Ensino Médio após 2025',
    4: 'Não concluí e não estou cursando o Ensino Médio'
}

df['situacao_conclusao'] = df['TP_ST_CONCLUSAO'].map(mapa_conclusao)

df['situacao_conclusao'].value_counts()

situacao_conclusao
Já concluí o Ensino Médio                               1888320
Estou cursando e concluirei o Ensino Médio em 2025      1811344
Estou cursando e concluirei o Ensino Médio após 2025      65958
Não concluí e não estou cursando o Ensino Médio           57500
Name: count, dtype: int64

In [94]:
tabela_st_conclusao_tipo_escola = (pd.crosstab(df['situacao_conclusao'], df['tipo_escola'], normalize='index') * 100).round(1)
tabela_st_conclusao_tipo_escola

tipo_escola,Não frequentou Ensino Médio,Pública + privada (com bolsa),Pública + privada (sem bolsa),Só privada (com bolsa),Só privada (sem bolsa),Só pública
situacao_conclusao,,,,,,
Estou cursando e concluirei o Ensino Médio após 2025,0.3,2.0,4.3,1.6,5.4,86.5
Estou cursando e concluirei o Ensino Médio em 2025,0.0,1.6,3.2,3.1,11.7,80.4
Já concluí o Ensino Médio,0.3,2.0,3.9,2.9,10.0,80.8
Não concluí e não estou cursando o Ensino Médio,8.9,1.5,4.0,0.6,2.1,82.9


O grupo que não concluiu e não está cursando o Ensino Médio apresenta uma característica própria: 8,9% desses candidatos relataram nunca ter frequentado o Ensino Médio, uma proporção muito acima das demais categorias (0,0% a 0,3%), o que sugere que parte desse grupo é composta por pessoas fora do sistema escolar regular, não apenas por quem interrompeu os estudos.

---